# Reproducción: TartuNLP @ AXOLOTL-24 (ruso)

Dorkin y Sirts (2024). Clasificador XLM-RoBERTa base + adaptador que decide si una **definición** describe un **uso** de la palabra. Es la misma idea que necesitas para la Subtask 2 de SemEval-2027.

| Métrica | TartuNLP (paper) | Aleatorio (referencia) |
|---|---|---|
| Macro-F1 | **0.64** | ≈0.35 |
| ARI | 0.098 | ≈0.00 |

El F1 es la métrica importante. El ARI de ruso es muy bajo para todos los equipos, así que es ruidoso.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → GPU.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 1. Código y datos

El código está en la rama `axolotl` (la rama principal es de otra tarea, SIGTYP). El original manda el modelo a la GPU de Mac (`mps`); se cambia a `cuda`.

In [2]:
%cd /content
!git clone -q -b axolotl https://github.com/slowwavesleep/ancient-lang-adapters.git
!git clone -q --depth 1 https://github.com/ltgoslo/axolotl24_shared_task.git
!sed -i 's/model.to("mps")/model.to("cuda")/' ancient-lang-adapters/predict_sense.py ancient-lang-adapters/predict_definition.py
!grep -n 'model.to' ancient-lang-adapters/predict_sense.py

/content
69:model.to("cuda")


## 2. Entorno con las versiones del paper

El repositorio fija versiones de finales de 2023 (torch 2.1.2, transformers 4.35.2, adapters 0.1.0) que no se instalan en el Python actual de Colab. Se crea un entorno aparte con Python 3.11. Tarda unos minutos porque torch es grande.

In [3]:
!pip install -q uv
!uv venv -q --python 3.11 /content/venv
!uv pip install -q --python /content/venv/bin/python adapters==0.1.0 transformers==4.35.2 datasets==2.15.0 evaluate==0.4.1 numpy==1.26.2 pandas==2.1.4 torch==2.1.2 tqdm==4.66.1 scikit-learn accelerate==0.25.0 huggingface-hub==0.19.4
!/content/venv/bin/python -c 'import torch, transformers, adapters; print(torch.__version__, transformers.__version__, adapters.__version__, "GPU:", torch.cuda.is_available())'

2.1.2+cu121 4.35.2 0.1.0 GPU: True


## 3. (Opcional, recomendado) Guardar el modelo en Drive

Si Colab se corta durante el entrenamiento, no pierdes los checkpoints.

In [5]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/axolotl_models
!ln -sfn /content/drive/MyDrive/axolotl_models /content/ancient-lang-adapters/trained_models

Mounted at /content/drive


## 4. Convertir los datos a pares (definición, uso)

Positivos: definición + uso de ese sentido (1). Negativos: la misma definición + usos de otros sentidos de la misma palabra (0). Formato: `definición<TAB>uso`.

Deberías ver 15997 pares de entrenamiento y 9851 de desarrollo.

In [6]:
%cd /content/ancient-lang-adapters
D = '/content/axolotl24_shared_task/data'
!/content/venv/bin/python convert_sense_data.py --in_path {D}/russian/axolotl.train.ru.tsv
!/content/venv/bin/python convert_sense_data.py --in_path {D}/russian/axolotl.dev.ru.tsv
!wc -l converted_data/*.json

/content/ancient-lang-adapters
   9851 converted_data/axolotl.dev.ru.json
  15997 converted_data/axolotl.train.ru.json
  25848 total


In [7]:
!uv pip install -q --python /content/venv/bin/python "pyarrow==14.0.2"
!/content/venv/bin/python -c "import datasets, numpy, pyarrow; print(datasets.__version__, numpy.__version__, pyarrow.__version__)"

2.15.0 1.26.2 14.0.2


Vemos como cambia el formato

In [8]:
!head -n 3 converted_data/axolotl.train.ru.json
!head -n 3 /content/axolotl24_shared_task/data/russian/axolotl.train.ru.tsv


{"text": "Казаться, являться пестрым, особенно издали, пестреть.\tСолнечный свет, падая струями сквозь узкие отверстия обветшалого намета, пестрил небольшими светлыми пятнами её косматую красно-гнедую шерсть.", "label": 0}
{"text": "остаться в живых, продолжать существовать, несмотря на неблагоприятные условия для жизни\tБарашкин всячески старался выжить из блиндажа переводчика Левина.", "label": 0}
{"text": "ездить на чём-либо для развлечения\tБудем кататься на слонах, встретим рассвет на какой-нибудь горе, среди лиан и орхидей.", "label": 1}
usage_id	word	orth	sense_id	gloss	example	indices_target_token	date	period
train_ru_0	миро	миро	miro_f2cnblJ7iBg	Миро, мvро ср., благовонное масло, пахучая масть или душистое маслянистое вещество.			old	old
train_ru_1	миро	миро	miro_8igZuzZK97Q	религ. жидкое ароматическое масло, освящаемое патриархом или митрополитом, используемое в таинстве миропомазания, при освящении престолов и антиминсов православных храмов, при венчании монарха на царство	З

## 5. Entrenar (configuración del paper para ruso)

XLM-R base, lr 5e-4, 50 épocas, batch 144, mejor checkpoint por F1 en desarrollo. En el paper tardó unas 3 h en una V100.

- **Si da error de memoria:** cambia a `--per_device_train_batch_size 72 --gradient_accumulation_steps 2` (mismo batch efectivo).
- **Si no tienes tiempo:** baja `--num_train_epochs` (por ejemplo a 15) y apúntalo como diferencia con el paper.
- **Si se cortó la sesión:** quita `--overwrite_output_dir` y vuelve a ejecutar; continúa desde el último checkpoint.

In [9]:
!/content/venv/bin/python train_sequence_classification.py \
  --model_name_or_path xlm-roberta-base \
  --task_name classification \
  --train_file converted_data/axolotl.train.ru.json \
  --validation_file converted_data/axolotl.dev.ru.json \
  --text_column_names text \
  --metric_name f1 \
  --do_train --do_eval \
  --learning_rate 5e-4 \
  --num_train_epochs 5 \
  --per_device_train_batch_size 144 \
  --per_device_eval_batch_size 256 \
  --evaluation_strategy epoch --save_strategy epoch \
  --load_best_model_at_end --metric_for_best_model f1 --save_total_limit 2 \
  --output_dir trained_models/ru --overwrite_output_dir \
  --report_to none
!cat trained_models/ru/eval_results.json

11/10/2026 08:02:21 - WARNING - __main__ - Process rank: 0, device: cuda:0, n_gpu: 1, distributed training: False, 16-bits training: False
11/10/2026 08:02:21 - INFO - __main__ - Training/evaluation parameters TrainingArguments(
_n_gpu=1,
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_pin_memory=True,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
dispatch_batches=None,
do_eval=True,
do_predict=False,
do_train=True,
eval_accumulation_steps=None,
eval_delay=0,
eval_steps=None,
evaluation_strategy=IntervalStrategy.EPOCH,
fp16=False,
fp16_backend=auto,
fp16_full_eval=False,
fp16_opt_level=O1,
fsdp=[],
fsdp_config={'min_num_params': 0, 'xla': False, 'xla_fsdp_grad_ckpt': False},
fsdp_min_num_params=0,
fsdp_t

## 6. Predecir sobre el test de ruso

Para cada uso nuevo se prueban todas las definiciones antiguas de la palabra. Si la mejor probabilidad es menor que 0.35 (umbral del paper, ya fijado en el script), el uso se marca como sentido nuevo.

In [10]:
!wc -l converted_data/*.json

   9851 converted_data/axolotl.dev.ru.json
  15997 converted_data/axolotl.train.ru.json
  25848 total


In [11]:
!/content/venv/bin/python predict_sense.py --model_name ru --base_model xlm-roberta-base \
  --in_path {D}/test/axolotl.test.ru.tsv --out_path pred.test.ru.tsv

Some weights of XLMRobertaAdapterModel were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['heads.default.3.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100% 31/31 [00:36<00:00,  1.19s/it]


## 7. Evaluar con el script oficial y comparar

Compara con la tabla del principio: F1 **0.64** y ARI 0.098. Con otra semilla o GPU es normal no clavar los decimales; lo que importa es quedar cerca y muy por encima del aleatorio (≈0.35).

In [12]:
!/content/venv/bin/python /content/axolotl24_shared_task/code/evaluation/scorer_track1.py \
  --gold {D}/russian/axolotl.test.ru.gold.tsv --pred pred.test.ru.tsv --output score.ru.txt 2>/dev/null
!cat score.ru.txt

ARI: 0.032
F1: 0.642
